
---

Si denotamos con el subíndice (c) a un componente de mezcla del conjunto de componentes (C), y denotamos el volumen de (c) usado en la mezcla como (x_c), el costo de la mezcla es

$$
\begin{align}
\text{costo} & = \sum_{c \in C} x_c P_c
\end{align}
$$

donde (P_c) es el precio por unidad de volumen del componente (c). Usando el diccionario de datos de Python definido anteriormente, el precio (P_c) está dado por `data[c]['cost']`.

### Restricción de Volumen

El requerimiento del cliente es producir un volumen total (V). Asumiendo soluciones ideales, la restricción se expresa como

$$
\begin{align}
V & = \sum_{c \in C} x_c
\end{align}
$$

donde (x_c) denota el volumen del componente (c) usado en la mezcla.

### Restricción de Composición del Producto

La composición del producto se especifica como 4% de alcohol en volumen. Denotando esto como (\bar{A}), la restricción se puede escribir como

$$
\begin{align}
\bar{A} & = \frac{\sum_{c \in C} x_c A_c}{\sum_{c \in C} x_c}
\end{align}
$$

donde (A_c) es el porcentaje de alcohol por volumen del componente (c). Tal como está escrita, esta es una restricción no lineal. Multiplicando ambos lados de la ecuación por el denominador se obtiene una restricción lineal:

$$
\begin{align}
\bar{A} \sum_{c \in C} x_c & = \sum_{c \in C} x_c A_c
\end{align}
$$

Una forma final de esta restricción puede darse de dos maneras. En la primera versión, restamos el lado izquierdo del derecho para obtener

$$
\begin{align}
0 & = \sum_{c \in C} x_c \left(A_c - \bar{A}\right) & \text{ Versión 1 de la restricción lineal de mezcla}
\end{align}
$$

Alternativamente, la suma en el lado izquierdo corresponde al volumen total. Dado que ese volumen es conocido como parte de la especificación del problema, la restricción de mezcla también se puede escribir como

$$
\begin{align}
\bar{A} V & = \sum_{c \in C} x_c A_c & \text{ Versión 2 de la restricción lineal de mezcla}
\end{align}
$$

¿Cuál usar? Cualquiera de las dos funciona bien en general. La ventaja de la versión 1 es que está completamente especificada por el requerimiento del producto (\bar{A}), lo cual a veces es útil al escribir código elegante en Python.

---


In [1]:
vol = 100
abv = 0.040

In [2]:
import pandas as pd
data = {
    'A': {'abv': 0.045, 'cost': 0.32},
    'B': {'abv': 0.037, 'cost': 0.25},
    'W': {'abv': 0.000, 'cost': 0.05},
}
df = pd.DataFrame(data).T
df

/Users/erickavendanogarcia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,abv,cost
A,0.045,0.32
B,0.037,0.25
W,0.000,0.05


### Crear modelo

In [3]:
from pyomo.environ import *
from pyomo.dae import *

model= ConcreteModel()

In [4]:
# Conjunto de componentes
model.Componentes = Set(initialize=df.index)

# Parámetros
model.pc = Param(model.Componentes, initialize=df['cost'].to_dict())
model.abv = Param(model.Componentes, initialize=df['abv'].to_dict())

# Variable indexada por el conjunto de componentes
model.x = Var(model.Componentes, domain=PositiveReals)

### Constricciones

In [5]:
def constriccion(model):
    return sum(model.x[i] for i in model.Componentes)== 100

model.constriccion= Constraint(rule=constriccion)

In [6]:
def constriccion1(model):
    return sum(model.x[i]*(model.abv[i] - abv) for i in model.Componentes) == 0

model.constriccion1 = Constraint(rule=constriccion1)


### Función Objetivo

In [7]:
def objetivo(model,i):
    return sum(model.x[i]*model.pc[i] for i in model.Componentes)
model.obj = Objective(rule=objetivo, sense=minimize)

### Resolver

In [8]:
solver = SolverFactory('glpk')
results = solver.solve(model, tee=True)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpkss5m8kp.glpk.raw
 --wglp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpis35ttdc.glpk.glp
 --cpxlp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpfjjot12k.pyomo.lp
Reading problem data from '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpfjjot12k.pyomo.lp'...
2 rows, 3 columns, 6 non-zeros
27 lines were read
Writing problem data to '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpis35ttdc.glpk.glp'...
18 lines were written
GLPK Simplex Optimizer 5.0
2 rows, 3 columns, 6 non-zeros
Preprocessing...
2 rows, 3 columns, 6 non-zeros
Scaling...
 A: min|aij| =  3.000e-03  max|aij| =  1.000e+00  ratio =  3.333e+02
GM: min|aij| =  5.233e-01  max|aij| =  1.911e+00  ratio =  3.651e+00
EQ: min|aij| =  2.739e-01  max|aij| =  1.000e+00  ratio =  3.651e+00
Constructing initial basis...
Size of triangular part is 1
      0: obj =   0.000000000e+00 i

In [9]:
model.obj.display()

obj : Size=1, Index=None, Active=True
    Key  : Active : Value
    None :   True : 27.625


In [10]:
model.x.display()

x : Size=3, Index=Componentes
    Key : Lower : Value : Upper : Fixed : Stale : Domain
      A :     0 :  37.5 :  None : False : False : PositiveReals
      B :     0 :  62.5 :  None : False : False : PositiveReals
      W :     0 :   0.0 :  None : False : False : PositiveReals
